# 05 — Model Training

**Goal:** Train Logistic Regression (baseline), Random Forest, and XGBoost models. Compare using stratified cross-validation. Select the best model for final evaluation.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import (
    load_raw_data, clean_data, separate_features_target, split_data,
    identify_column_types, build_preprocessor, TARGET_COL
)
from src.feature_engineering import engineer_features
from src.train import (
    cross_validate_models, train_final_model,
    save_model, save_comparison_report
)
from src.evaluate import (
    compute_metrics, print_classification_report,
    plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve,
    plot_model_comparison, find_optimal_threshold, generate_evaluation_summary
)

## 5.1 Prepare Data

In [ ]:
# Load, clean, and engineer features
df_raw = load_raw_data()
df = clean_data(df_raw)
df = engineer_features(df)

# Separate and split
X, y, ids = separate_features_target(df)
X_train, X_test, y_train, y_test = split_data(X, y)

# Build preprocessor
numeric_cols, categorical_cols = identify_column_types(X_train)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

print(f"\nReady for training:")
print(f"  Training set: {X_train.shape}")
print(f"  Test set: {X_test.shape} (untouched until final evaluation)")
print(f"  Numeric features: {len(numeric_cols)}")
print(f"  Categorical features: {len(categorical_cols)}")

## 5.2 Cross-Validation Comparison

In [ ]:
comparison_df = cross_validate_models(X_train, y_train, preprocessor)
comparison_df

In [ ]:
# Save comparison
save_comparison_report(comparison_df)

# Plot comparison
fig = plot_model_comparison(comparison_df)
plt.show()

## 5.3 Select and Train Final Model

Based on cross-validation results, we select the model with the best balance of recall (identifying churners) and precision (avoiding false alarms). We choose XGBoost as it typically offers the best ROC-AUC and F1 balance.

In [ ]:
# Select model based on best F1 score from CV
best_model_name = comparison_df.loc[comparison_df['f1_mean'].idxmax(), 'Model']
print(f"Selected model: {best_model_name}")
print(f"  CV F1: {comparison_df.loc[comparison_df['f1_mean'].idxmax(), 'f1_mean']:.4f}")
print(f"  CV ROC-AUC: {comparison_df.loc[comparison_df['f1_mean'].idxmax(), 'roc_auc_mean']:.4f}")

In [ ]:
# Train final model on full training set
pipeline = train_final_model(X_train, y_train, preprocessor, model_name=best_model_name)

# Save the model
save_model(pipeline)

## 5.4 Find Optimal Threshold (on Training Data)

In [ ]:
# Use training predictions to find threshold (NOT test set)
y_train_prob = pipeline.predict_proba(X_train)[:, 1]
optimal_threshold, best_f1 = find_optimal_threshold(y_train, y_train_prob, metric='f1')
print(f"\nOptimal threshold: {optimal_threshold:.2f} (Training F1: {best_f1:.4f})")

## 5.5 Final Evaluation on Test Set

This is the **one-time** evaluation on the held-out test set.

In [ ]:
# Generate test predictions
y_test_prob = pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= optimal_threshold).astype(int)

# Compute metrics
test_metrics = compute_metrics(y_test, y_test_pred, y_test_prob, threshold=optimal_threshold)
print("\n" + "="*50)
print("FINAL TEST SET EVALUATION")
print("="*50)
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Classification report
print("\nDetailed Classification Report:")
print_classification_report(y_test, y_test_pred)

In [ ]:
# Confusion Matrix
fig = plot_confusion_matrix(y_test, y_test_pred, title=f'Confusion Matrix — {best_model_name}')
plt.show()

In [ ]:
# ROC Curve
fig = plot_roc_curve(y_test, y_test_prob, title=f'ROC Curve — {best_model_name}')
plt.show()

In [ ]:
# Precision-Recall Curve
fig = plot_precision_recall_curve(y_test, y_test_prob, title=f'Precision-Recall Curve — {best_model_name}')
plt.show()

In [ ]:
# Generate evaluation summary markdown
generate_evaluation_summary(test_metrics, model_name=best_model_name)

## 5.6 Notes

- The test set was used **only once** for final evaluation.
- The classification threshold was chosen on training data.
- Model selection was based on cross-validation scores, not test performance.
- Results reflect the model's performance on this specific dataset split and may vary with different seeds or data.